In [2]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────────
!pip install timm diffusers accelerate datasets fvcore -q

import os, sys
if not os.path.exists('DiT'):
    !git clone https://github.com/facebookresearch/DiT.git
sys.path.insert(0, os.path.abspath('DiT'))

import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers.models import AutoencoderKL
from models import DiT_models

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

CKPT = "DiT-XL-2-256x256.pt"
if not os.path.exists(CKPT):
    !wget -q https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt

dit = DiT_models["DiT-XL/2"](input_size=32).to(device)
state = torch.load(CKPT, map_location=device)
dit.load_state_dict(state.get("model", state))
dit.eval()
for p in dit.parameters():
    p.requires_grad = False

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

hidden_dim = dit.x_embedder.proj.out_channels
print(f"DiT hidden dim: {hidden_dim}")
print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Cloning into 'DiT'...
remote: Enumerating objects: 102, done.
remote: Total 102 (delta 0), reused 0 (delta 0), pack-reused 102 (from 1)
Receiving objects: 100% (102/102), 6.37 MiB | 13.17 MiB/s, done.
Resolving deltas: 100% (55/55), done.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

DiT hidden dim: 1152
Setup complete.


In [3]:
# ── Cell 2: Real Latent Cache (tiny-imagenet, no auth required) ───────────────
from datasets import load_dataset
from torchvision import transforms

LATENT_CACHE_FILE = "imagenet_latents_cache.pt"
N_CACHE      = 512
ENCODE_BATCH = 16

if os.path.exists(LATENT_CACHE_FILE):
    cache        = torch.load(LATENT_CACHE_FILE)
    latent_cache = cache["latents"]
    label_cache  = cache["labels"]
    print(f"Loaded cached latents: {latent_cache.shape}")
else:
    print("Building latent cache from tiny-imagenet...")
    # Public, no gating, no trust_remote_code needed
    ds = load_dataset("zh-plus/tiny-imagenet", split="valid", streaming=True)

    tf = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])

    all_latents, all_labels = [], []
    batch_imgs, batch_labels = [], []

    for sample in ds:
        img = sample["image"].convert("RGB")
        batch_imgs.append(tf(img))
        batch_labels.append(sample["label"])

        if len(batch_imgs) == ENCODE_BATCH:
            imgs_t = torch.stack(batch_imgs).to(device)
            with torch.no_grad():
                lat = vae.encode(imgs_t).latent_dist.sample() * 0.18215
            all_latents.append(lat.cpu())
            all_labels.extend(batch_labels)
            batch_imgs, batch_labels = [], []
            if len(all_labels) >= N_CACHE:
                break

    latent_cache = torch.cat(all_latents)[:N_CACHE]
    label_cache  = torch.tensor(all_labels[:N_CACHE])
    torch.save({"latents": latent_cache, "labels": label_cache}, LATENT_CACHE_FILE)
    print(f"Cached {latent_cache.shape[0]} latents → {LATENT_CACHE_FILE}")

Building latent cache from tiny-imagenet...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

Cached 512 latents → imagenet_latents_cache.pt


In [ ]:
# ── Cell 3: TokenPredictor + PruningWrapper ────────────────────────────────────

class TokenPredictor(nn.Module):
    def __init__(self, d_model: int, bottleneck: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, bottleneck),
            nn.GELU(),
            nn.Linear(bottleneck, 1),
        )

    def forward(self, x: torch.Tensor, temperature: float = 1.0):
        logits    = self.mlp(x)
        pair      = torch.cat([logits, -logits], dim=-1)
        hard      = F.gumbel_softmax(pair, tau=temperature, hard=True)
        keep_mask = hard[..., 1]   # [B, N]  — still has grad_fn via straight-through
        return keep_mask, logits


class PruningWrapper(nn.Module):
    def __init__(self, block: nn.Module, predictor: TokenPredictor):
        super().__init__()
        self.block           = block
        self.predictor       = predictor
        self.pruning_enabled = True
        self._last_mask      = None   # detached, for logging only
        self._live_mask      = None   # keeps grad_fn, used for sparsity loss
        self._temperature    = 1.0

    def forward(self, x: torch.Tensor, c: torch.Tensor):
        if not self.pruning_enabled:
            self._live_mask = None
            return self.block(x, c)

        B, N, D = x.shape

        keep_mask, logits  = self.predictor(x, self._temperature)
        self._live_mask    = keep_mask          # grad_fn intact
        self._last_mask    = keep_mask.detach() # for logging

        keep_idx = keep_mask[0].detach().bool()
        N_kept   = keep_idx.sum().item()

        if N_kept == 0 or N_kept == N:
            return self.block(x, c)

        idx      = keep_idx.nonzero(as_tuple=True)[0]       # [N_kept]
        x_kept   = x[:, idx, :]                             # [B, N_kept, D]
        out_kept = self.block(x_kept, c)                    # [B, N_kept, D]

        idx_exp  = idx[None, :, None].expand(B, N_kept, D)
        out      = x.scatter(1, idx_exp, out_kept)
        return out


def set_pruning_mode(model: nn.Module, enabled: bool):
    for block in model.blocks:
        if isinstance(block, PruningWrapper):
            block.pruning_enabled = enabled

def get_all_masks(model: nn.Module, live: bool = False):
    """live=True returns masks with grad_fn for loss; False returns detached for logging."""
    attr = "_live_mask" if live else "_last_mask"
    return [
        getattr(b, attr) for b in model.blocks
        if isinstance(b, PruningWrapper) and getattr(b, attr) is not None
    ]


# Wrap layers
PRUNING_LAYERS = [4, 8, 12, 16, 20, 24]

for idx in PRUNING_LAYERS:
    # Peel ALL wrapper layers, however deep
    original = dit.blocks[idx]
    while isinstance(original, PruningWrapper):
        original = original.block
    predictor = TokenPredictor(hidden_dim, bottleneck=64).to(device)
    dit.blocks[idx] = PruningWrapper(original, predictor)

all_parasites = [b.predictor for b in dit.blocks if isinstance(b, PruningWrapper)]
n_params = sum(p.numel() for pp in all_parasites for p in pp.parameters())
print(f"Pruning layers : {PRUNING_LAYERS}")
print(f"Parasite params: {n_params:,}  ({n_params/1e6:.2f}M)")


# ── Cell 4: Training Loop ──────────────────────────────────────────────────────
from diffusion import create_diffusion
from tqdm import tqdm

TARGET_RATIO   = 0.5
LAMBDA_DISTILL = 1.0
LAMBDA_SPARSE  = 0.5
STEPS          = 5000
BATCH_SIZE     = 4
LR             = 1e-4
TEMP_START     = 2.0
TEMP_END       = 0.5

diffusion = create_diffusion(timestep_respacing="")

optimizer = torch.optim.Adam(
    [p for pp in all_parasites for p in pp.parameters()], lr=LR
)

history = {"loss_distill": [], "loss_sparse": [], "keep_ratio": []}

pbar = tqdm(range(STEPS))
for step in pbar:
    optimizer.zero_grad()

    frac        = step / max(STEPS - 1, 1)
    temperature = TEMP_START * (TEMP_END / TEMP_START) ** frac
    for pp in all_parasites:
        pp._temperature = temperature

    idx_batch = torch.randint(0, latent_cache.shape[0], (BATCH_SIZE,))
    x0 = latent_cache[idx_batch].to(device)
    y  = label_cache[idx_batch].to(device)

    t     = torch.randint(0, 1000, (BATCH_SIZE,), device=device)
    noise = torch.randn_like(x0)
    z_t   = diffusion.q_sample(x0, t, noise=noise)

    # ── Teacher pass: fully isolated, no grad ─────────────────────────────
    set_pruning_mode(dit, enabled=False)
    dit.eval()
    with torch.no_grad():
        teacher_out = dit(z_t, t, y)

    # ── Student pass: grad flows through Gumbel-Softmax to predictors ─────
    set_pruning_mode(dit, enabled=True)
    dit.train()
    student_out = dit(z_t, t, y)

    # live masks retain grad_fn from Gumbel-Softmax straight-through
    live_masks  = get_all_masks(dit, live=True)
    log_masks   = get_all_masks(dit, live=False)

    # distill loss: grad path is student_out → scatter → block → x_kept → predictor mask
    loss_distill = F.mse_loss(student_out, teacher_out)

    # sparse loss: directly on live keep_mask (has grad_fn)
    live_ratio   = torch.stack([m.float().mean() for m in live_masks]).mean()
    loss_sparse  = (live_ratio - TARGET_RATIO) ** 2

    total_loss = LAMBDA_DISTILL * loss_distill + LAMBDA_SPARSE * loss_sparse
    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        [p for pp in all_parasites for p in pp.parameters()], max_norm=1.0
    )
    optimizer.step()

    # logging (detached)
    log_ratio = torch.stack([m.float().mean() for m in log_masks]).mean().item()
    history["loss_distill"].append(loss_distill.item())
    history["loss_sparse"].append(loss_sparse.item())
    history["keep_ratio"].append(log_ratio)

    pbar.set_description(
        f"distill={loss_distill.item():.4f} | "
        f"sparse={loss_sparse.item():.4f} | "
        f"kept={log_ratio*100:.1f}% | "
        f"temp={temperature:.2f}"
    )

print("Training complete.")



Pruning layers : [4, 8, 12, 16, 20, 24]
Parasite params: 456,966  (0.46M)


distill=0.0027 | sparse=0.0001 | kept=49.3% | temp=1.69:  12%|█▏        | 613/5000 [05:49<41:26,  1.76it/s]

In [ ]:
# ── Cell 5: Training Curves ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["loss_distill"])
axes[0].set_title("Distillation Loss (MSE)")
axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["loss_sparse"], color="orange")
axes[1].axhline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_title("Sparsity Loss")
axes[1].set_xlabel("Step")
axes[1].grid(True, alpha=0.3)

axes[2].plot([r * 100 for r in history["keep_ratio"]], color="green")
axes[2].axhline(TARGET_RATIO * 100, color="red", linestyle="--",
                label=f"Target {TARGET_RATIO*100:.0f}%")
axes[2].set_title("Token Keep Ratio (%)")
axes[2].set_xlabel("Step")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: FLOP Benchmark ────────────────────────────────────────────────────
from fvcore.nn import FlopCountAnalysis

z_bench = torch.randn(1, 4, 32, 32, device=device)
t_bench = torch.tensor([500], device=device)
y_bench = torch.tensor([207], device=device)

dit.eval()
set_pruning_mode(dit, enabled=False)
flops_teacher = FlopCountAnalysis(dit, (z_bench, t_bench, y_bench))
flops_teacher.unsupported_ops_warnings(False)
gmac_teacher  = flops_teacher.total() / 1e9

set_pruning_mode(dit, enabled=True)
for pp in all_parasites:
    pp.eval()
flops_student = FlopCountAnalysis(dit, (z_bench, t_bench, y_bench))
flops_student.unsupported_ops_warnings(False)
gmac_student  = flops_student.total() / 1e9

reduction = (1 - gmac_student / gmac_teacher) * 100
print(f"Teacher GMACs : {gmac_teacher:.2f}")
print(f"Student GMACs : {gmac_student:.2f}")
print(f"FLOP Reduction: {reduction:.1f}%")

In [ ]:
# ── Cell 7: Qualitative Comparison ────────────────────────────────────────────
import numpy as np

def to_img(tensor):
    img = tensor[0].permute(1, 2, 0).cpu().float().numpy()
    return np.clip((img + 1) / 2, 0, 1)

def run_inference(class_label=207, seed=42, ddim_steps=50):
    torch.manual_seed(seed)
    z    = torch.randn(1, 4, 32, 32, device=device)
    y    = torch.tensor([class_label], device=device)
    diff = create_diffusion(timestep_respacing=f"ddim{ddim_steps}")
    dit.eval()

    set_pruning_mode(dit, enabled=False)
    with torch.no_grad():
        lat_t = diff.ddim_sample_loop(dit, shape=(1, 4, 32, 32),
                    noise=z.clone(), model_kwargs={"y": y},
                    device=device, progress=False)
        px_t = vae.decode(lat_t / 0.18215).sample

    set_pruning_mode(dit, enabled=True)
    for pp in all_parasites: pp.eval()
    with torch.no_grad():
        lat_s = diff.ddim_sample_loop(dit, shape=(1, 4, 32, 32),
                    noise=z.clone(), model_kwargs={"y": y},
                    device=device, progress=False)
        px_s = vae.decode(lat_s / 0.18215).sample

    masks = get_all_masks(dit)
    mse   = F.mse_loss(px_s.float(), px_t.float()).item()
    kept  = sum(m.mean().item() for m in masks) / len(masks) * 100
    return px_t, px_s, masks, mse, kept

CLASS_LABEL = 208
SEED        = 42
GRID        = 16   # 16x16 = 256 tokens for DiT-XL/2 at latent_size=32

px_t, px_s, masks, mse, kept = run_inference(CLASS_LABEL, SEED)

n_layers = len(masks)
fig, axes = plt.subplots(2, max(n_layers, 2), figsize=(4 * max(n_layers, 2), 9))

axes[0, 0].imshow(to_img(px_t))
axes[0, 0].set_title(f"Teacher (full)\nClass {CLASS_LABEL}"); axes[0, 0].axis('off')

axes[0, 1].imshow(to_img(px_s))
axes[0, 1].set_title(f"Student (pruned)\n{kept:.1f}% kept | MSE {mse:.5f}"); axes[0, 1].axis('off')

for ax in axes[0, 2:]: ax.axis('off')

for li, mask in enumerate(masks):
    grid = mask[0].cpu().float().numpy().reshape(GRID, GRID)
    im   = axes[1, li].imshow(grid, cmap='magma', interpolation='nearest', vmin=0, vmax=1)
    axes[1, li].set_title(f"Layer {PRUNING_LAYERS[li]}\n{mask[0].mean().item()*100:.0f}% kept")
    axes[1, li].axis('off')
    plt.colorbar(im, ax=axes[1, li], shrink=0.7)

plt.suptitle(f"Seed {SEED} | Class {CLASS_LABEL} | Pixel MSE: {mse:.5f}")
plt.tight_layout()
plt.show()

In [ ]:
# Real latency benchmark
starter = torch.cuda.Event(enable_timing=True)
ender   = torch.cuda.Event(enable_timing=True)
REPS    = 50

# Teacher
set_pruning_mode(dit, enabled=False)
dit.eval()
with torch.no_grad():
    for _ in range(10): dit(z_bench, t_bench, y_bench)  # warmup
    starter.record()
    for _ in range(REPS): dit(z_bench, t_bench, y_bench)
    ender.record()
torch.cuda.synchronize()
ms_teacher = starter.elapsed_time(ender) / REPS

# Student
set_pruning_mode(dit, enabled=True)
with torch.no_grad():
    for _ in range(10): dit(z_bench, t_bench, y_bench)
    starter.record()
    for _ in range(REPS): dit(z_bench, t_bench, y_bench)
    ender.record()
torch.cuda.synchronize()
ms_student = starter.elapsed_time(ender) / REPS

print(f"Teacher: {ms_teacher:.2f} ms")
print(f"Student: {ms_student:.2f} ms")
print(f"Speedup: {ms_teacher/ms_student:.3f}x")